# Laya Decision Models with OpenVINO™

[Laya](https://huggingface.co/convaiinnovations/laya) is a **multilingual, non-autoregressive System 1 decision model** from Convai Innovations. Give it a **state** (an email, a support ticket, or any text/JSON document) and a set of **typed questions**; it returns typed answers with mathematically calibrated probabilities in a **single forward pass**, across 100+ languages.

Unlike a generative LLM, Laya never produces free text, so there is nothing to parse and nothing to hallucinate. It was trained with reinforcement learning against strictly proper scoring rules (**RLCD** — Reinforcement Learning for Calibrated Decisions), which makes reporting *honest* probabilities the only way to maximize reward.

Key features:

* **Three typed questions** — `choice` (routing, moderation), `score` (urgency, severity) and `noul` (a yes/no with a probability) — all answered in one batched forward pass.
* **Calibrated probabilities** — RLCD training rewards honest uncertainty, so the reported probabilities can be used directly in thresholds and guardrails.
* **Non-generative** — structured output only: no prompt parsing, no hallucination, deterministic latency.
* **Fast and lightweight** — a 421M-parameter bidirectional encoder (ModernBERT-large) with a small transformer decision head.

Typical application scenarios: email and support-ticket routing, guardrails and moderation, urgency/severity scoring, spam and phishing detection, churn-risk flags — anywhere an LLM pipeline would otherwise parse a generated answer out of free text.

More details in the original [model card](https://huggingface.co/convaiinnovations/laya), [GitHub repository](https://github.com/NandhaKishorM/laya) and [documentation](https://nandhakishorm.github.io/laya/).

<p>
    <img src="https://raw.githubusercontent.com/NandhaKishorM/laya/main/assets/long_context_8192.png" alt="Laya long-context accuracy" width="90%"/>
</p>

In this notebook you will:

1. Install a build of `optimum-intel` that carries the Laya integration (fork branch `laya`);
2. Download a Laya checkpoint from the Hugging Face Hub;
3. Convert it to the OpenVINO IR format;
4. Run inference with the official `system_one` API and triage emails interactively.

## Table of contents:
- [Prerequisites](#Prerequisites)
- [Download the model](#Download-the-model)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
- [Run inference](#Run-inference)
- [Interactive demo](#Interactive-demo)

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Fetch notebook_utils and cmd_helper modules
import requests
from pathlib import Path

for module in ("cmd_helper.py", "notebook_utils.py", "pip_helper.py"):
    if not Path(module).exists():
        r = requests.get(url=f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{module}")
        Path(module).write_text(r.text, encoding="utf-8")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("laya-decision-models.ipynb")


### Install Dependencies
[back to top ⬆️](#Table-of-contents:)

Laya support is not in an optimum-intel release yet, so this notebook installs a fork that carries the integration (branch [`laya`](https://github.com/xiangfuxwang/optimum-intel-dev/tree/laya)):

```
git+https://github.com/xiangfuxwang/optimum-intel-dev.git@laya
```

Everything else installs from PyPI as usual.

In [ ]:
from pip_helper import pip_install

# Remove pre-installed versions to avoid conflicts
%pip uninstall -q -y optimum optimum-intel optimum-onnx

# torch AND torchvision must come from the PyTorch CPU index as a matched pair:
# optimum-intel depends on torchvision, and installing it later from PyPI would pull a
# CUDA build that crashes `import transformers` when paired with CPU-only torch.
pip_install(
    "-q",
    "torch>=2.1",
    "torchvision",
    "--extra-index-url", "https://download.pytorch.org/whl/cpu",
)
pip_install(
    "-q",
    "openvino>=2025.4.0",
    "openvino-tokenizers",
    "nncf>=3.3.0",
    "optimum>=1.24.0",
    "transformers>=4.53.0",
    "safetensors",
    "huggingface_hub",
    "ipywidgets",
)
# Laya support for optimum-intel: install the fork branch until it is upstreamed
pip_install("-q", "git+https://github.com/xiangfuxwang/optimum-intel-dev.git@laya")

## Download the model
[back to top ⬆️](#Table-of-contents:)

Three checkpoints are available, all Apache 2.0 licensed:

| Checkpoint | Backbone | Parameters | Context | Notes |
|---|---|---|---|---|
| [`convaiinnovations/laya`](https://huggingface.co/convaiinnovations/laya) | ModernBERT-large | 421M | 512 | English (default) |
| [`convaiinnovations/laya-multilingual`](https://huggingface.co/convaiinnovations/laya-multilingual) | mmBERT-base (22 layers, 256k vocab) | 322M | 1024 (up to 8192) | 100+ languages |
| [`convaiinnovations/laya-typed-decisions`](https://huggingface.co/convaiinnovations/laya-typed-decisions) | ModernBERT-large | 421M | 1024 | fine-tuned for typed decisions (0.766 accuracy) |

This notebook uses the English checkpoint by default; the same code works for all three.

A Laya checkpoint is not a standard `transformers` model: the weights live in a single `model.safetensors`, the head hyper-parameters and calibration temperatures in `rl_agent_config.json`, and the tokenizer in a `tokenizer/` subfolder.

In [ ]:
import ipywidgets as widgets
from huggingface_hub import snapshot_download

# Users with slow access to huggingface.co can uncomment the next line to use the HF mirror:
# import os; os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

model_options = {
    "laya (English)": "convaiinnovations/laya",
    "laya-multilingual": "convaiinnovations/laya-multilingual",
    "laya-typed-decisions (fine-tuned)": "convaiinnovations/laya-typed-decisions",
}

model_selector = widgets.Dropdown(options=list(model_options.items()), description="Checkpoint:")
display(model_selector)

In [ ]:
model_dir = Path(model_selector.value.split("/")[-1])

if not (model_dir / "rl_agent_config.json").exists():
    snapshot_download(repo_id=model_selector.value, local_dir=model_dir)

print(f"Checkpoint downloaded to: {model_dir.resolve()}")
print(f"Files: {sorted(p.name for p in model_dir.iterdir())}")

## Convert the model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

The model is converted with `optimum-cli export openvino`, producing a single OpenVINO IR:

* **Inputs**: `input_ids`, `attention_mask` `[batch, sequence]`; `marker_pos`, `marker_mask` `[batch, num_options]`; `qtype` `[batch]` (0=choice, 1=score, 2=noul)
* **Outputs**: `logits` `[batch, num_options]` — one score per option marker — and `act_logits` `[batch, 2]` (the answer-vs-escalate head)

Each answer option is prefixed by a `[MASK]` marker token in the input sequence; `marker_pos` points at those markers and the decision head scores exactly these positions.

The export also saves the assets the runtime needs next to the IR: `rl_agent_config.json` (calibration temperatures, sequence budgets) and the `tokenizer/` folder, plus the OpenVINO tokenizer/detokenizer IR.

Equivalent command-line invocation:

```bash
optimum-cli export openvino --model convaiinnovations/laya --task feature-extraction --weight-format fp16 laya-ov-fp16
```

### Compress model weights
[back to top ⬆️](#Table-of-contents:)

By default the exported weights keep the checkpoint's fp32 precision (~1.6 GB). The `--weight-format` option compresses them at export time via [NNCF](https://github.com/openvinotoolkit/nncf) weight compression:

| Format | IR size | Notes |
|---|---|---|
| `fp16` (default below) | ~0.8 GB | measured on the example below: identical decisions, maximum probability deviation < 0.0002 |
| `int8` | smaller | further reduces footprint; validate decisions on your own data |
| `int4` | smallest | most aggressive compression; validate decisions on your own data |

Because Laya returns calibrated probabilities that often feed thresholds and guardrails, verify that decisions remain acceptable after `int8`/`int4` compression before using them in production.


In [ ]:
import ipywidgets as widgets

# Weight compression applied at export time (see the table above)
weight_format_options = {
    "FP16 (default)": "fp16",
    "INT8": "int8",
    "INT4": "int4",
}

weight_format_selector = widgets.Dropdown(
    options=list(weight_format_options.items()), value="fp16", description="Weights:"
)
display(weight_format_selector)


In [ ]:
from cmd_helper import optimum_cli

weight_format = weight_format_selector.value
ov_model_dir = Path(f"{model_dir.name}-ov-{weight_format}")

if not (ov_model_dir / "openvino_model.xml").exists():
    print("Exporting to OpenVINO IR, this may take a few minutes ...")
    optimum_cli(model_dir, ov_model_dir, additional_args={"task": "feature-extraction", "weight-format": weight_format})
else:
    print(f"Found existing conversion in {ov_model_dir}/, skipping export")

print(f"Files: {sorted(p.name for p in ov_model_dir.iterdir())}")


### Select inference device
[back to top ⬆️](#Table-of-contents:)

OpenVINO offers different inference devices: CPU, GPU, NPU, etc. Intel GPUs (iGPU/dGPU) work well with the compressed IR produced above — the weights are already stored in a low-precision format, so no conversion happens at load time. NPU is not validated for this model and is excluded from the widget below. The [latency measurement](#Measure-inference-latency) section shows how to compare devices on your machine.


In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
display(device)


In [ ]:
from optimum.intel.openvino.modeling_laya import OVModelForLayaDecision

ov_model = OVModelForLayaDecision.from_pretrained(ov_model_dir, device=device.value)

print(f"Inputs : {list(ov_model.input_names)}")
print(f"Outputs: {list(ov_model.output_names)}")

## Run inference
[back to top ⬆️](#Table-of-contents:)

The runtime exposes the official `system_one(state, questions)` API, so the converted model is a drop-in replacement for the PyTorch original. A question is defined by:

* `type` — `"choice"`, `"score"` or `"noul"`;
* `instructions` — the question text;
* `criteria` — the options: a dict for `choice`, an ordered list of levels for `score`, and optional true/false descriptions for `noul`.

The example below is the classic Laya use case, **email triage**: route the email to the right team, flag spam, and score urgency — all in one forward pass.

In [ ]:
import json

state = {
    "subject": "Invoice #4382 overdue - service suspension warning",
    "body": (
        "Hi, our accounting team flagged invoice #4382 (issued Sep 12, $1,240) as unpaid. "
        "Payment was sent on Sep 15 via wire transfer, please check on your side. "
        "If the payment is not located by Friday the service will be suspended."
    ),
}

questions = {
    "category": {
        "type": "choice",
        "instructions": "Which team should handle the email in `body`?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, integrations",
            "sales": "pricing, demos, new purchases",
            "other": "none of the above",
        },
    },
    "is_spam": {"type": "noul", "instructions": "Is this email unsolicited spam or bulk marketing?"},
    "urgency": {
        "type": "score",
        "instructions": "How urgent is the issue described in `body`?",
        "criteria": ["no time pressure", "needs attention soon", "blocking issue or hard deadline"],
    },
}

result = ov_model.system_one(state, questions)
print(json.dumps(result, indent=2, ensure_ascii=False))

Reading the output:

* `choice` questions return the selected option plus calibrated `probabilities` for every option — here `billing` at ~96%;
* `score` questions return the expected level (`score`) over the ordered `criteria` — here `1.79` on a `0..2` scale, close to *blocking issue*;
* `noul` questions return the probability that the statement holds (`noul: 0.03` → almost certainly not spam);
* `confidence` is `1 − normalized entropy` of the answer distribution;
* `rl_agent.act_probability` is the model's own estimate that it *should* answer rather than escalate;
* `usage.input_tokens` counts the tokens actually processed — all questions ran in a single forward pass.

The next cell shows a compact way to consume the results for a second, very different email:

In [ ]:
state2 = {
    "subject": "App crashes when opening settings",
    "body": (
        "Hello, since the last update the Android app force-closes every time I open "
        "the settings screen. Reinstalling did not help. I use a Pixel 8 with Android 15."
    ),
}

result2 = ov_model.system_one(state2, questions)

for qid, ans in result2["answers"].items():
    if ans["type"] == "choice":
        top = ans["choice"]
        print(f"{qid:10s} -> {top} (p={ans['probabilities'][top]:.1%})")
    elif ans["type"] == "score":
        print(f"{qid:10s} -> {ans['score']:.2f} / {len(ans['legend']) - 1}")
    else:
        print(f"{qid:10s} -> {ans['noul']:.1%}")

### Measure inference latency
[back to top ⬆️](#Table-of-contents:)

The cell below measures the average latency of a `system_one` call for the example above. A few warm-up calls are excluded from the measurement, because the first inference on each device includes one-time model compilation (and, on GPU, kernel JIT) overhead. To compare devices, re-run the [device selection](#Select-inference-device) cell with a different choice and execute the subsequent cells again.


In [ ]:
import time

for _ in range(3):  # warm-up: excludes one-time compile/JIT overhead
    ov_model.system_one(state, questions)

n_runs = 10
start = time.perf_counter()
for _ in range(n_runs):
    ov_model.system_one(state, questions)
avg_ms = (time.perf_counter() - start) / n_runs * 1e3

print(f"Device: {device.value}")
print(f"Average latency over {n_runs} runs: {avg_ms:.1f} ms per system_one call")


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Paste any email body below and triage it with the same question set.

**Mind the input length limit.** The serialized state must fit into the model's context window — 512 tokens for the English `laya` checkpoint (1024 for `laya-typed-decisions`, 1024–8192 for `laya-multilingual`). When it does not fit, the runtime silently truncates the **end** of the serialized state to the remaining token budget, so details at the end of a long email (deadlines, contact info, signatures) may be dropped. Keep inputs concise and put the important facts first — the demo prints the number of input tokens actually processed after each run.


In [ ]:
from IPython.display import clear_output

body_input = widgets.Textarea(
    value=state["body"], description="", layout=widgets.Layout(width="100%", height="140px")
)
run_button = widgets.Button(description="Triage email", button_style="primary")
truncation_note = widgets.HTML(
    "<b>Note:</b> inputs that exceed the model's context window are truncated at the end — "
    "keep the email concise and put important details first."
)
demo_output = widgets.Output()


def triage(_):
    with demo_output:
        clear_output()
        res = ov_model.system_one({"subject": "(user input)", "body": body_input.value}, questions)
        for qid, ans in res["answers"].items():
            if ans["type"] == "choice":
                top = ans["choice"]
                print(f"{qid:10s}: {top}  (p={ans['probabilities'][top]:.1%})")
            elif ans["type"] == "score":
                print(f"{qid:10s}: {ans['score']:.2f} / {len(ans['legend']) - 1}")
            else:
                print(f"{qid:10s}: {ans['noul']:.1%}")
        print(f"({res['usage']['input_tokens']} input tokens processed)")


run_button.on_click(triage)
display(widgets.VBox([widgets.Label("Email body:"), body_input, truncation_note, run_button, demo_output]))


In [ ]:
# # Uncomment to free disk space (removes the downloaded checkpoint and the converted IR)
# import shutil

# shutil.rmtree(model_dir, ignore_errors=True)
# shutil.rmtree(ov_model_dir, ignore_errors=True)